In [6]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough


In [ ]:
import argparse, inspect, json, os, pickle, socket, subprocess, warnings, random, math, librosa, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from sklearn.metrics import accuracy_score, auc, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score, roc_curve
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorboard.backend.event_processing import event_accumulator
from torch.amp import GradScaler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from transformers import AutoConfig, AutoFeatureExtractor, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.simplefilter("ignore", UserWarning)

import torchaudio
import random
import pandas as pd

def generate_random_segments(file_list, sr=16000, min_sec=0.4, max_sec=0.6, repeat=5):
    records = []

    for path in file_list:
        wav, fs = torchaudio.load(path)
        if fs != sr:
            wav = torchaudio.functional.resample(wav, fs, sr)
        wav = wav.squeeze(0)
        length = wav.shape[0]

        for _ in range(repeat):
            dur = random.uniform(min_sec, max_sec)
            seg_len = int(dur * sr)

            if seg_len >= length:
                start = 0
            else:
                start = random.randint(0, length - seg_len)

            end = start + seg_len

            records.append({
                "path_file": path,
                "index_start": start,
                "index_end": end,
            })

    return pd.DataFrame(records)

In [68]:
df_cough_train = pd.read_csv('/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.train')
df_cough_test = pd.read_csv('/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/metadata.csv.val')
df_cough_train = df_cough_train[['path_file']]
df_cough_test = df_cough_test[['path_file']]

df_noise_speech = pd.read_csv("/run/media/fourier/Data1/Pras/Interspeech2025/RIRS_NOISES/data_augmentation_noises_speechs_labels.tsv", delimiter="\t", header=None)

df_noise_speech_train, df_noise_speech_test = train_test_split(df_noise_speech, test_size=0.2, stratify=df_noise_speech[1])
df_noise_speech_train = generate_random_segments(df_noise_speech_train[0].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.6, repeat=6)
df_noise_speech_test = generate_random_segments(df_noise_speech_test[0].values.tolist(), sr=16000, min_sec=0.4, max_sec=0.6, repeat=6)

df_cough_train = df_cough_train.reset_index(drop=True)
df_cough_test = df_cough_test.reset_index(drop=True)

df_noise_speech_train = df_noise_speech_train.reset_index(drop=True)
df_noise_speech_test = df_noise_speech_test.reset_index(drop=True)

df_cough_train['index_start'] = 0
df_cough_train['index_end'] = -1
df_cough_train['label'] = 1
df_cough_test['index_start'] = 0
df_cough_test['index_end'] = -1
df_cough_test['label'] = 1

df_noise_speech_train['label'] = 0
df_noise_speech_test['label'] = 0

df_train = pd.concat([df_cough_train, df_noise_speech_train], axis=0, ignore_index=True)
df_test = pd.concat([df_cough_test, df_noise_speech_test], axis=0, ignore_index=True)

In [69]:
df_train = pd.concat([df_cough_train, df_noise_speech_train], axis=0, ignore_index=True)
df_test = pd.concat([df_cough_test, df_noise_speech_test], axis=0, ignore_index=True)

In [73]:
df_train.to_csv(f"/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cough_detection.csv.train", index=False)
df_test.to_csv(f"/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/data/cough_detection.csv.test", index=False)